## TPOT + MLflow AutoML

**Platform:** TPOT (open-source AutoML) + MLflow (experiment tracking) 
**Notebook Content** Tasks 1–3 (setup, configuration, all-features AutoML run)

In [3]:
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Project root is one level above notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tpot import TPOTRegressor

# #region agent log
import tpot as _tpot
_dbg("D", "tpot import success", {"tpot_version": getattr(_tpot, "__version__", "?"), "tpot_file": _tpot.__file__, "has_TPOTRegressor": hasattr(_tpot, "TPOTRegressor")})
# #endregion

import config
from preprocess import get_feature_matrix, load_and_clean, train_test_split_xy

config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
mlflow.set_experiment(config.MLFLOW_EXPERIMENT_NAME)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MLflow URI:", config.MLFLOW_TRACKING_URI)


/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PROJECT_ROOT: /Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3
MLflow URI: file:///Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/mlruns


/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/.venv/lib/python3.9/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")


### Task 1 — Dataset Loading and Setup

#### Documentation

| Item | Detail |
|---|---|
| **Source** | Provided `athletes.csv` |
| **Version used** | **Cleaned** data via `preprocess.load_and_clean()` — not raw |
| **Target** | `total_lift = deadlift + candj + snatch + backsq` |
| **Preprocessing** | Assign 2 filters: weight/height/age/gender bounds, lift caps, replace `Decline to answer\|` with NaN, `dropna()` |
| **Leakage rule** | Lift components and `total_lift` are excluded from predictors |
| **Split** | 80/20 train/test, `random_state=42` |

In [4]:
clean = load_and_clean()

print("Cleaned shape:", clean.shape)
print("\nDtypes:")
print(clean.dtypes)
print("\nNull counts:")
print(clean.isna().sum())
print("\nSample rows:")
display(clean.head())
print("\nTarget describe:")
display(clean[config.TARGET].describe())

Cleaned shape: (47061, 12)

Dtypes:
athlete_id    float64
gender         object
age           float64
height        float64
weight        float64
howlong        object
background     object
deadlift      float64
candj         float64
snatch        float64
backsq        float64
total_lift    float64
dtype: object

Null counts:
athlete_id    0
gender        0
age           0
height        0
weight        0
howlong       0
background    0
deadlift      0
candj         0
snatch        0
backsq        0
total_lift    0
dtype: int64

Sample rows:


,athlete_id,gender,age,height,weight,howlong,background,deadlift,candj,snatch,backsq,total_lift
0,6491.0,Male,37.0,73.0,230.0,4+ years|,I played youth or high school level sports|,435.0,265.0,200.0,414.0,1314.0
1,8242.0,Male,40.0,68.0,177.0,2-4 years|,I played youth or high school level sports|I p...,365.0,225.0,185.0,365.0,1140.0
2,11416.0,Male,31.0,65.0,150.0,2-4 years|,I played youth or high school level sports|I p...,465.0,290.0,225.0,405.0,1385.0
3,21053.0,Male,42.0,72.0,210.0,4+ years|,I played youth or high school level sports|,515.0,325.0,235.0,505.0,1580.0
4,21269.0,Male,30.0,71.0,200.0,1-2 years|,I played youth or high school level sports|I p...,385.0,235.0,175.0,315.0,1110.0



Target describe:


count    47061.000000
mean      1006.854062
std        277.909926
min          4.000000
25%        791.000000
50%       1030.000000
75%       1210.000000
max       2280.000000
Name: total_lift, dtype: float64

In [5]:
X, y, feature_cols = get_feature_matrix(clean, config.ALL_FEATURES)
X_train, X_test, y_train, y_test = train_test_split_xy(X, y)

print("All features:", feature_cols)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Lift components in features?", any(c in feature_cols for c in config.LIFT_COMPONENTS))
print("Ready for AutoML:", X_train.isna().sum().sum() == 0 and y_train.isna().sum() == 0)

All features: ['age', 'height', 'weight', 'gender_male', 'is_experienced', 'has_athletic_background']
X_train: (37648, 6) | X_test: (9413, 6)
Lift components in features? False
Ready for AutoML: True


### Task 2 — Chosen MLOps Platform AutoML Configuration

#### Platform: **TPOT + MLflow**

#### Why this stack is appropriate

1. Aligns with lecture open-source AutoML options (TPOT / H2O / Auto-sklearn) while keeping H2O for the required Task 9 repeat.
2. Runs fully **local** — no cloud account — consistent with Assign 2’s reproducible local MLOps pattern.
3. TPOT returns an exportable sklearn pipeline; MLflow captures what the search actually did.

#### AutoML settings (from `config.py`)

In [6]:
tpot_settings = {
    "target": config.TARGET,
    "features": feature_cols,
    "holdout": f"{int((1 - config.TEST_SIZE) * 100)}/{int(config.TEST_SIZE * 100)} train/test",
    "random_state": config.SEED,
    "scoring": "neg_mean_squared_error  (primary report metric: RMSE)",
    "cv": config.TPOT_CV,
    "generations": config.TPOT_GENERATIONS,
    "population_size": config.TPOT_POPULATION_SIZE,
    "max_time_mins": config.TPOT_MAX_TIME_MINS,
    "n_jobs": config.TPOT_N_JOBS,
    "algorithm_space": "TPOT default sklearn regressor config (trees, linear, ensembles, selectors, scalers)",
    "excluded_features": list(config.LIFT_COMPONENTS) + [config.TARGET],
}
pd.Series(tpot_settings, name="value").to_frame()

,value
target,total_lift
features,"[age, height, weight, gender_male, is_experien..."
holdout,80/20 train/test
random_state,42
scoring,neg_mean_squared_error (primary report metric...
cv,5
generations,5
population_size,20
max_time_mins,10
n_jobs,-1


### Validation strategy

- **Outer holdout:** 80/20 train/test (`SEED=42`) for final reported RMSE / MAE / R².
- **Inner AutoML validation:** TPOT uses `cv`-fold cross-validation on the training set (`neg_mean_squared_error`) to rank pipelines during search.

### Task 3 — AutoML Run Using All Features

In [7]:
tpot = TPOTRegressor(
    generations=config.TPOT_GENERATIONS,
    population_size=config.TPOT_POPULATION_SIZE,
    cv=config.TPOT_CV,
    scoring="neg_mean_squared_error",
    max_time_mins=config.TPOT_MAX_TIME_MINS,
    random_state=config.SEED,
    verbosity=2,
    n_jobs=config.TPOT_N_JOBS,
)

fit_start = time.perf_counter()
tpot.fit(X_train, y_train)
fit_seconds = time.perf_counter() - fit_start

y_pred = tpot.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print(f"Best pipeline:\n{tpot.fitted_pipeline_}")
print(f"\nHoldout RMSE: {rmse:.4f}")
print(f"Holdout MAE:  {mae:.4f}")
print(f"Holdout R2:   {r2:.4f}")
print(f"TPOT fit wall-clock (s): {fit_seconds:.1f}")

Version 0.12.2 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


Optimization Progress:   0%|          | 0/20 [00:00<?, ?pipeline/s]

/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/.venv/lib/python3.9/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3


Generation 1 - Current best internal CV score: -24484.912144159665

Generation 2 - Current best internal CV score: -24484.912144159665

Generation 3 - Current best internal CV score: -24484.912144159665

Generation 4 - Current best internal CV score: -24437.685182146688

Generation 5 - Current best internal CV score: -24437.685182146688

Best pipeline: RandomForestRegressor(input_matrix, bootstrap=True, max_features=0.4, min_samples_leaf=16, min_samples_split=14, n_estimators=100)
Best pipeline:
Pipeline(steps=[('randomforestregressor',
                 RandomForestRegressor(max_features=0.4, min_samples_leaf=16,
                                       min_samples_split=14,
                                       random_state=42))])

Holdout RMSE: 157.6968
Holdout MAE:  120.9107
Holdout R2:   0.6832
TPOT fit wall-clock (s): 128.4


In [8]:
def tpot_leaderboard(tpot_model: TPOTRegressor) -> pd.DataFrame:
    """Convert TPOT evaluated_individuals_ into a ranked leaderboard."""
    rows = []
    for pipeline, meta in tpot_model.evaluated_individuals_.items():
        cv_score = meta.get("internal_cv_score", np.nan)
        rows.append(
            {
                "pipeline": pipeline,
                "cv_neg_mse": cv_score,
                "cv_rmse": float(np.sqrt(-cv_score)) if pd.notna(cv_score) and cv_score < 0 else np.nan,
                "generation": meta.get("generation", np.nan),
                "operator_count": meta.get("operator_count", np.nan),
            }
        )
    lb = pd.DataFrame(rows)
    # Higher neg_mse is better; sort descending then by simpler pipelines
    lb = lb.sort_values(["cv_neg_mse", "operator_count"], ascending=[False, True]).reset_index(drop=True)
    lb.index = lb.index + 1
    lb.index.name = "rank"
    return lb


leaderboard_all = tpot_leaderboard(tpot)
leaderboard_path = config.REPORTS_DIR / "tpot_leaderboard_all.csv"
leaderboard_all.to_csv(leaderboard_path)

print(f"Leaderboard rows: {len(leaderboard_all)} -> {leaderboard_path}")
print("\nTop 10 by CV score:")
display(leaderboard_all.head(10))

Leaderboard rows: 112 -> /Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign3/reports/tpot_leaderboard_all.csv

Top 10 by CV score:


,pipeline,cv_neg_mse,cv_rmse,generation,operator_count
rank,,,,,
1,"RandomForestRegressor(input_matrix, RandomFore...",-24437.685182,156.325574,4,1
2,"RandomForestRegressor(input_matrix, RandomFore...",-24480.655151,156.462951,5,1
3,"RandomForestRegressor(input_matrix, RandomFore...",-24480.655151,156.462951,5,1
4,RandomForestRegressor(ElasticNetCV(input_matri...,-24484.912144,156.476555,0,2
5,RandomForestRegressor(ElasticNetCV(input_matri...,-24484.912144,156.476555,5,2
6,RandomForestRegressor(ElasticNetCV(input_matri...,-24484.912144,156.476555,5,2
7,"RandomForestRegressor(input_matrix, RandomFore...",-24574.862997,156.763717,4,1
8,RandomForestRegressor(RidgeCV(RobustScaler(inp...,-24578.296758,156.774669,3,3
9,RandomForestRegressor(RidgeCV(RobustScaler(inp...,-24578.296758,156.774669,5,3


In [9]:
best_pipeline_str = str(tpot.fitted_pipeline_)
pipeline_export_path = config.REPORTS_DIR / "tpot_best_pipeline_all.py"
tpot.export(str(pipeline_export_path))

with mlflow.start_run(run_name="tpot_all_features") as run:
    mlflow.log_params(
        {
            "tool": "tpot",
            "feature_set": "all",
            "n_features": len(feature_cols),
            "features": ",".join(feature_cols),
            "generations": config.TPOT_GENERATIONS,
            "population_size": config.TPOT_POPULATION_SIZE,
            "cv": config.TPOT_CV,
            "max_time_mins": config.TPOT_MAX_TIME_MINS,
            "seed": config.SEED,
            "scoring": "neg_mean_squared_error",
        }
    )
    mlflow.log_metrics(
        {
            "holdout_rmse": rmse,
            "holdout_mae": mae,
            "holdout_r2": r2,
            "fit_seconds": fit_seconds,
            "n_pipelines_evaluated": float(len(leaderboard_all)),
        }
    )
    mlflow.log_text(best_pipeline_str, "best_pipeline.txt")
    mlflow.log_artifact(str(leaderboard_path))
    mlflow.log_artifact(str(pipeline_export_path))
    run_id = run.info.run_id

print("MLflow run_id:", run_id)
print("Best model (pipeline):", best_pipeline_str)
print(f"Primary validation metric (holdout RMSE): {rmse:.4f}")

MLflow run_id: 61b2a0c0b4a0428fbffeef5ef3af58f8
Best model (pipeline): Pipeline(steps=[('randomforestregressor',
                 RandomForestRegressor(max_features=0.4, min_samples_leaf=16,
                                       min_samples_split=14,
                                       random_state=42))])
Primary validation metric (holdout RMSE): 157.6968


#### Task 3 summary

- **Best model:** `RandomForestRegressor` (`max_features=0.4`, `min_samples_leaf=16`, `min_samples_split=14`)
- **Holdout metrics:** RMSE **157.70**, MAE **120.91**, R² **0.683** (CV RMSE **156.33**)
- **Artifacts:** `reports/tpot_leaderboard_all.csv`, `reports/tpot_best_pipeline_all.py`; MLflow run `tpot_all_features`
